In [1]:
import numpy as np
from sklearn.datasets import fetch_openml
from tqdm import tqdm
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='liac-arff')

In [4]:
NUM_INPUT = 784
NUM_OUTPUT = 500
DT = 0.005
T_MAX = 0.350
NUM_IMAGES = 60000

def assign_labels():
    print("Loading weights and training subset via sklearn...")
    weights = np.load('vdsp_weights.npy')
    
    images = mnist.data[:NUM_IMAGES] / 255.0
    labels = mnist.target[:NUM_IMAGES].astype(int)
    
    spike_counts = np.zeros((NUM_OUTPUT, 10))
    v_pre = np.zeros(NUM_INPUT)
    v_post = np.zeros(NUM_OUTPUT)
    
    # Track adaptation and clamps just like in training
    n_adapt = np.zeros(NUM_OUTPUT)
    refractory_pre = np.zeros(NUM_INPUT)
    refractory_post = np.zeros(NUM_OUTPUT)
    wta_clamp = np.zeros(NUM_OUTPUT)
    
    for img_idx, img in tqdm(enumerate(images), total=len(images), desc="Labeling"):
        v_pre.fill(0.0)
        v_post.fill(0.0)
        refractory_pre.fill(0.0)
        refractory_post.fill(0.0)
        wta_clamp.fill(0.0)
        img_label = labels[img_idx]
        
        for t in np.arange(0, T_MAX, DT):

            refractory_pre -= DT
            refractory_post -= DT
            wta_clamp -= DT
            
            # Input Neurons
            active_pre = refractory_pre <= 0
            v_pre[active_pre] += (DT / 0.03) * (-v_pre[active_pre] + img[active_pre] + 0.5)
            
            spikes_pre = v_pre >= 1.0
            v_pre[spikes_pre] = -1.0
            refractory_pre[spikes_pre] = 0.005
            
            # Output Neurons
            active_post = (refractory_post <= 0) & (wta_clamp <= 0)
            I_post = spikes_pre.astype(float) @ weights
            
            v_post[active_post] += (DT / 0.03) * (-v_post[active_post] + I_post[active_post] - n_adapt[active_post])
            n_adapt -= (DT / 1.0) * n_adapt
            
            spikes_post = v_post >= 1.0
            
            # Enforce Winner-Take-All during labeling
            if spikes_post.any():
                winner_idx = np.argmax(v_post)
                
                # Record the spike for the true winner only!
                spike_counts[winner_idx, img_label] += 1
                
                v_post[winner_idx] = 0.0
                refractory_post[winner_idx] = 0.005
                n_adapt[winner_idx] += 0.01
                
                wta_clamp[:] = 0.010
                wta_clamp[winner_idx] = 0.0
                v_post[wta_clamp > 0] = 0.0
                
    assigned_labels = np.argmax(spike_counts, axis=1)
    np.save('assigned_labels.npy', assigned_labels)
    print(f"Assigned Labels: {assigned_labels}")

In [5]:
assign_labels()

Loading weights and training subset via sklearn...


Labeling: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 60000/60000 [17:22<00:00, 57.57it/s]


Assigned Labels: [2 8 5 7 5 5 7 2 2 1 5 3 4 4 9 3 2 3 3 1 2 4 8 8 6 8 0 4 9 5 3 2 6 4 5 6 6
 5 6 7 3 9 9 9 2 5 6 9 8 6 3 3 4 4 3 7 9 5 8 2 3 5 8 6 2 8 5 2 4 3 9 9 5 7
 7 1 3 3 8 9 1 8 4 4 4 7 8 9 7 9 5 3 3 3 6 7 3 2 4 4 4 5 1 9 5 8 1 4 4 4 8
 6 8 4 2 9 9 2 1 1 8 1 2 3 1 0 4 5 8 6 5 0 1 2 2 7 8 8 7 2 1 5 6 3 3 8 1 2
 8 5 3 8 3 7 5 3 2 9 6 8 4 8 9 8 6 7 2 3 8 4 8 5 4 8 8 3 4 8 4 2 3 5 3 4 2
 7 4 9 9 1 5 4 4 0 8 5 7 8 8 4 4 1 9 8 5 5 4 4 4 8 0 4 3 2 4 7 5 9 1 8 9 9
 9 6 2 0 8 3 3 7 9 4 5 8 4 2 2 4 8 1 8 9 4 4 9 3 4 4 9 5 7 8 6 8 1 3 2 4 9
 5 2 5 4 4 3 4 7 6 8 8 5 3 1 3 6 3 0 0 6 9 8 6 3 2 9 2 3 8 8 7 7 9 9 8 8 7
 8 8 2 7 7 2 4 6 5 9 9 4 3 2 2 1 3 8 5 4 8 8 8 7 2 4 7 7 8 8 5 7 5 4 4 3 2
 5 9 9 7 0 3 4 4 2 8 8 2 9 2 9 5 8 5 8 9 3 8 5 5 8 3 9 2 8 8 7 2 8 9 4 2 8
 7 9 6 6 9 5 1 3 6 3 8 9 5 8 7 8 8 6 7 8 7 8 8 0 5 5 5 3 7 9 2 4 7 7 3 8 8
 2 5 5 7 2 0 5 2 9 8 5 1 7 8 4 9 3 2 8 6 3 1 5 6 7 5 3 7 4 5 3 9 5 4 8 5 7
 6 1 6 8 9 7 7 4 2 7 2 5 8 6 7 5 3 8 1 4 3 4 0 8 4 8 8 6 5 3 8 4 6 9 7 1 3
 5 3 3 4